<a href="https://colab.research.google.com/github/SimonHtet/Hotel/blob/main/exceltis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'Exeltis' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv


In [ ]:
# == section 2: profile/explore ==
# markdown cell above this ##2.profiling -finding data quality issues

#structure: are the data types right?--
df.info()
df.head()

#-- whats missing?
print(df.isnull().sum())
print()
print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print('Full duplicate rows:', df.duplicated().sum())
print('Duplicate order_id:', df['order_id'].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 2499 entries, 0 to 2499
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             2499 non-null   object        
 1   order_date           2498 non-null   datetime64[ns]
 2   customer_name        2499 non-null   object        
 3   client_segment       2499 non-null   object        
 4   client_city          2499 non-null   object        
 5   country              2499 non-null   object        
 6   client_singup_date   2480 non-null   datetime64[ns]
 7   product_name         2498 non-null   object        
 8   quantity             2499 non-null   int64         
 9   unit_price           2499 non-null   float64       
 10  unit_cost            2498 non-null   float64       
 11  discount_pct         2498 non-null   float64       
 12  product_category     2498 non-null   object        
 13  product_subcategory  2498 non-null   o

In [ ]:
df[['quantity','unit_price','unit_cost','discount_pct']].describe()

,quantity,unit_price,unit_cost,discount_pct
count,2499.000000,2499.000000,2498.000000,2498.000000
mean,50.026411,38.215714,19.329215,0.171737
std,28.813891,20.897281,11.705823,0.101216
min,1.000000,2.010000,-5.000000,0.000000
25%,24.000000,19.905000,10.660000,0.080000
50%,50.000000,37.940000,17.770000,0.170000
75%,75.000000,55.880000,29.070000,0.260000
max,100.000000,74.990000,39.970000,0.350000


In [ ]:
print('Rows where cost == price :', (df['unit_cost'] == df['unit_cost']).sum())
print('Rows with inactive product sold :', (df['product_active'] == False).sum())

Rows where cost == price : 2498
Rows with inactive product sold : 508


In [ ]:
print(df['sales_channel'].value_counts())
print(df[['order_date','client_singup_date']].head())

sales_channel
Online       845
Phone        842
Sales Rep    812
Name: count, dtype: int64
  order_date client_singup_date
0 2024-09-13         2022-04-05
1 2023-01-27         2021-06-25
2 2023-03-19         2024-11-02
3 2024-09-06         2022-12-26
4 2024-01-15         2024-12-26


In [ ]:
def parse_mixed_dates(val):
    try:
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(val)))
    except:
        pass
    for fmt in ('%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

df['client_singup_date'] = df['client_singup_date'].apply(parse_mixed_dates).dt.strftime('%Y/%m/%d')
print(df['client_singup_date'].isna().sum())
print(df[['order_date','client_singup_date']].head())

19
  order_date client_singup_date
0 2024-09-13         2022/04/05
1 2023-01-27         2021/06/25
2 2023-03-19         2024/11/02
3 2024-09-06         2022/12/26
4 2024-01-15         2024/12/26


In [ ]:
print(df['order_date'].iloc[80])

2023-05-24 00:00:00


In [ ]:
df[df['order_date'] == '31-02-2024']

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce').dt.strftime('%Y/%m/%d')
print(df['order_date'].isna().sum())  # see how many became NaT

1


In [ ]:
df[df.duplicated(keep=False)] #finding duplicated rows

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [ ]:
df = df.drop_duplicates() #drop duplicates

In [ ]:
print('duplicate order_ids:', df['order_id'].duplicated().sum())
print('rows now:',len(df))


duplicate order_ids: 0
rows now: 2499


In [ ]:
df[df['discount_pct']>1]

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep


In [ ]:
df.loc[df['discount_pct']>1, 'discount_pct'] = pd.NA

In [ ]:
print('discounts > 1 now:', (df['discount_pct']>1).sum())

discounts > 1 now: 0


In [ ]:
df = df.copy()
df['country'] = df['country'].replace('0', 'Unknown')
df['client_segment'] = df['client_segment'].replace('0', 'Unknown')
print(df['country'].value_counts())
print(df['client_segment'].value_counts())

country
France      729
Portugal    675
Italy       588
Spain       506
Unknown       1
Name: count, dtype: int64
client_segment
Distributor    826
Retail         562
Hospitality    561
Supermarket    532
Unknown         18
Name: count, dtype: int64


In [ ]:
print(df.shape)
print(df['order_date'].dtype)
print((df['country']=='0').sum())

(2499, 18)
object
0


In [ ]:
import pandas as pd
import numpy as mp

print('row:', len(df))
print('dup order_id:', df['order_id'].duplicated().sum())
print('min quantity:', df['quantity'].min())
print('discount >1:', (df['discount_pct']>1).sum())
print("country '0':", (df['country'] == '0').sum(),"segment '0':",(df['client_segment']=='0').sum())

row: 2499
dup order_id: 0
min quantity: 1
discount >1: 0
country '0': 0 segment '0': 0


In [ ]:
rejects_qty = df[df['quantity'] <= 0].copy() #record of what i removed
df = df[df['quantity'] > 0]
print('min quantity:', df['quantity'].min())
print('row:', len(df))
print('quarantined:', len(rejects_qty))

min quantity: 1
row: 2499
quarantined: 0


In [ ]:
rejects_qty = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
rejects_qty = rejects_qty[rejects_qty['quantity'] <= 0]
print(rejects_qty)

   order_id      order_date customer_name client_segment client_city country  \
22   O00023  9/16/2023 0:00      Chen Inc    Distributor  Barnesport   Spain   

   client_singup_date     product_name  quantity  unit_price  unit_cost  \
22              45688  Milk Product 61       -10       26.87      12.92   

    discount_pct product_category product_subcategory           supplier_name  \
22          0.34            Dairy                Milk  Smith, Graham and Rowe   

   product_active sales_channel       sales_rep  
22           True         Phone  Lindsey Curtis  


In [ ]:
assert df['order_id'].is_unique, "order_id must be unique"
assert (df['quantity']>0).all(), "quantity must be positive"
assert df['discount_pct'].dropna().between(0,1).all(), "discount must be 0-1"
assert (df['country'] != '0').all(), "no '0' placeholders in country"
assert (df['client_segment'] != '0').all(), "no '0' placeholders in segments"
assert (df['unit_price'] > 0).all(), "unit price must be positive"
assert df['order_date'].dtype == 'datetime64[ns]', "order_date must be datetime"
assert df['client_singup_date'].dtype == 'datetime64[ns]', "client_singup_date must be datetime"

print(" All validation checks passed -", len(df), "clean rows")

 All validation checks passed - 2499 clean rows


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y/%m/%d')
df['client_singup_date'] = pd.to_datetime(df['client_singup_date'])
print(df['order_date'].dtype)
print(df['order_date'].head())
print(df['client_singup_date'].dtype)
print(df['client_singup_date'].head())

datetime64[ns]
0   2024-09-13
1   2023-01-27
2   2023-03-19
3   2024-09-06
4   2024-01-15
Name: order_date, dtype: datetime64[ns]
datetime64[ns]
0   2022-04-05
1   2021-06-25
2   2024-11-02
3   2022-12-26
4   2024-12-26
Name: client_singup_date, dtype: datetime64[ns]


In [ ]:
print('df rows:', len(df))
print('unique customers:',df['customer_name'].nunique())
print('dim_customer rows:', len(dim_customer))
print('dim_customer rows:', len(dim_customer))
print('columns:', df.columns.tolist())

df rows: 2499
unique customers: 120
dim_customer rows: 121
dim_customer rows: 121
columns: ['order_id', 'order_date', 'customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date', 'product_name', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'product_category', 'product_subcategory', 'supplier_name', 'product_active', 'sales_channel', 'sales_rep']


In [ ]:
# dim customer - one row per unique customer
cust_cols = ['customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date']
dim_customer = (
    df[cust_cols]
    .drop_duplicates(subset=['customer_name'])  # one row per customer
    .reset_index(drop=True)
)

dim_customer['client_singup_date'] = dim_customer['client_singup_date'].dt.normalize()
dim_customer['customer_key'] = dim_customer.index + 1

print('dim_customer rows:', len(dim_customer))
dim_customer.head()

dim_customer rows: 120


,customer_name,client_segment,client_city,country,client_singup_date,customer_key
0,Mccoy Ltd,Hospitality,New Douglas,Portugal,2022-04-05,1
1,Porter and Sons,Distributor,Jorgefort,France,2021-06-25,2
2,Day Group,Distributor,Courtneystad,Portugal,2024-11-02,3
3,"Clark, Farrell and Sheppard",Distributor,New Molly,France,2022-12-26,4
4,Hawkins and Sons,Distributor,Barnetthaven,France,2024-12-26,5


In [ ]:
# ==== dim_product
prod_cols = ['product_name','product_category', 'product_subcategory','supplier_name','product_active','unit_cost']
dim_product = (
    df[prod_cols]
    .drop_duplicates(subset=['product_name'])  # one row per product
    .reset_index(drop=True)
)

dim_product['product_key'] = dim_product.index + 1
print('dim_product rows:',len(dim_product))

# ==== dim_channel
dim_channel = (
    df[['sales_channel']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_channel['channel_key'] = dim_channel.index + 1
print('dim_channel rows:',len(dim_channel))
dim_product.head()

#dim_date
dim_date = (
    df[['order_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_date['date_key'] = dim_date.index + 1
dim_date['year'] = dim_date['order_date'].dt.year
dim_date['quarter'] = dim_date['order_date'].dt.quarter
dim_date['month'] = dim_date['order_date'].dt.month
dim_date['month_name'] = dim_date['order_date'].dt.month_name()
dim_date['day_name'] = dim_date['order_date'].dt.day_name()
print('dim_date rows:',len(dim_date))
dim_date.head()


dim_product rows: 80
dim_channel rows: 3
dim_date rows: 710


,order_date,date_key,year,quarter,month,month_name,day_name
0,2024-09-13,1,2024.0,3.0,9.0,September,Friday
1,2023-01-27,2,2023.0,1.0,1.0,January,Friday
2,2023-03-19,3,2023.0,1.0,3.0,March,Sunday
3,2024-09-06,4,2024.0,3.0,9.0,September,Friday
4,2024-01-15,5,2024.0,1.0,1.0,January,Monday


In [ ]:
# Sales the transaction
fact_sales = df.copy()

# join each table to bring in keys
fact_sales = fact_sales.merge(dim_customer[['customer_name', 'customer_key']], on='customer_name', how='left')
fact_sales = fact_sales.merge(dim_product[['product_name', 'product_key']], on='product_name', how='left')
fact_sales = fact_sales.merge(dim_channel[['sales_channel', 'channel_key']], on='sales_channel', how='left')
fact_sales = fact_sales.merge(dim_date[['order_date', 'date_key']], on='order_date', how='left')

#compute the measures
fact_sales['revenue'] = fact_sales['quantity'] * fact_sales['unit_price'] *(1 - fact_sales['discount_pct'].fillna(0))
fact_sales['cost'] = fact_sales['quantity'] * fact_sales['unit_cost'].round(2)
fact_sales['profit'] = fact_sales['revenue'] - fact_sales['cost'].round(2)

#order_id + keys+transactions measures
fact_cols = ['order_id', 'order_date', 'date_key', 'customer_key', 'product_key', 'channel_key', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'revenue', 'cost', 'profit']
fact_sales = fact_sales[fact_cols]

print('fact_sales rows:',len(fact_sales))
fact_sales

fact_sales rows: 2499


,order_id,order_date,date_key,customer_key,product_key,channel_key,quantity,unit_price,unit_cost,discount_pct,revenue,cost,profit
0,O00001,2024-09-13,1,1,1,1,33,57.98,12.48,0.06,1798.5396,411.84,1386.6996
1,O00002,2023-01-27,2,2,2,2,21,72.42,12.20,0.04,1459.9872,256.20,1203.7872
2,O00003,2023-03-19,3,3,3,1,49,30.96,7.57,0.25,1137.7800,370.93,766.8500
3,O00004,2024-09-06,4,4,4,3,90,58.39,38.22,0.30,3678.5700,3439.80,238.7700
4,O00005,2024-01-15,5,5,5,3,6,27.35,35.41,0.15,139.4850,212.46,-72.9750
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2494,O02496,2024-06-01,596,85,4,2,20,69.60,38.22,0.24,1057.9200,764.40,293.5200
2495,O02497,2023-03-25,218,67,39,3,52,32.61,8.88,0.12,1492.2336,461.76,1030.4736
2496,O02498,2023-11-18,560,65,24,3,62,71.24,9.69,0.23,3400.9976,600.78,2800.2176
2497,O02499,2023-02-01,163,72,7,2,56,13.25,39.97,0.17,615.8600,2238.32,-1622.4600
